# Imports

In [1]:

import torch as torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset
import torchvision.models as models
from torchvision import transforms
from tqdm.notebook import tqdm
import pandas as pd
import numpy as np
import cv2
import os
import time
import random
import gc
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report
from PIL import Image
from collections import OrderedDict
import shutil

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# utils.py

In [2]:
# Hyperparams
HEIGHT = 128
WIDTH = 128
CHANNELS = 3
FPS = 10
DURATION = 3
SEQ_LEN = FPS * DURATION

# MVO Prediction Logic Mapping
# FRONT: 0, LEFT: 1, RIGHT: 2
def get_label_id(label_name):
    mapping = {'front': 0, 'left': 1, 'right': 2}
    return mapping.get(label_name.lower(), 0)

# Data Paths
VIDEO_DIR = '/content/drive/MyDrive/test_data/videos/'
LABEL_DIR = '/content/drive/MyDrive/test_data/labels/'

# Cache Directory - Use local SSD for 10-100x faster I/O
CACHE_DIR = '/content/cache/'  # Local SSD (fast, cleared on runtime disconnect)
# Alternative: '/content/drive/MyDrive/test_data/frame_cache/' (slow, but persistent)

# Incomplete_EluSEEdate_Dataset
# Complete Final Training Datasets

# Check if paths exist to avoid errors later
if not os.path.exists(VIDEO_DIR):
    print(f"WARNING: Directory not found: {VIDEO_DIR}")
    print("Please update the VIDEO_DIR variable in this cell.")

if not os.path.exists(LABEL_DIR):
    print(f"WARNING: Directory not found: {LABEL_DIR}")
    print("Please update the LABEL_DIR variable in this cell.")

# preprocessor.py

In [3]:
def preprocess_and_cache_videos(video_folder, cache_folder, transforms, num_frames=30):
    """
    Pre-extract and cache video frames for faster training.

    Args:
        video_folder: Path to video files
        cache_folder: Path to save cached frames
        transforms: Torchvision transforms to apply
        num_frames: Number of frames per video (default: 30)
    """
    # Create cache directory if it doesn't exist
    os.makedirs(cache_folder, exist_ok=True)

    # Get all video files
    video_files = [f for f in os.listdir(video_folder) if f.endswith('.mp4')]

    print(f"Starting frame extraction for {len(video_files)} videos...")
    print(f"Cache directory: {cache_folder}")
    print(f"This may take a few minutes but only needs to be done once.\n")

    start_time = time.time()
    cached_count = 0
    skipped_count = 0

    for video_name in tqdm(video_files, desc="Preprocessing videos"):
        video_path = os.path.join(video_folder, video_name)
        cache_path = os.path.join(cache_folder, video_name.replace('.mp4', '.npy'))

        # Skip if already cached
        if os.path.exists(cache_path):
            skipped_count += 1
            continue

        # Extract frames from video
        cap = cv2.VideoCapture(video_path)
        frames = []

        for _ in range(num_frames):
            ret, frame = cap.read()
            if not ret:
                # Pad with black frame if video is shorter
                frame_tensor = torch.zeros((3, HEIGHT, WIDTH))
            else:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                if transforms:
                    frame = Image.fromarray(frame)
                    frame_tensor = transforms(frame)
                else:
                    frame_tensor = torch.from_numpy(frame).permute(2, 0, 1).float() / 255.0

            frames.append(frame_tensor)

        cap.release()

        # Stack and save as numpy array
        video_tensor = torch.stack(frames, dim=0)  # [30, 3, 128, 128]
        np.save(cache_path, video_tensor.numpy())

        cached_count += 1
        del frames, video_tensor

    elapsed_time = time.time() - start_time

    print(f"\n✓ Preprocessing completed!")
    print(f"  - Newly cached: {cached_count} videos")
    print(f"  - Already cached: {skipped_count} videos")
    print(f"  - Total time: {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")
    print(f"  - Cache size: ~{(cached_count + skipped_count) * 2:.0f} MB")
    print(f"\n✓ Ready for training! The dataset will now load from cache.")

preprocessing_transforms = transforms.Compose([
    transforms.Resize((HEIGHT, WIDTH)),
    transforms.ToTensor()
])

# Uncomment below only if you want to pre-cache ALL videos upfront:
preprocess_and_cache_videos(VIDEO_DIR, CACHE_DIR, preprocessing_transforms)

Starting frame extraction for 2000 videos...
Cache directory: /content/cache/
This may take a few minutes but only needs to be done once.



Preprocessing videos:   0%|          | 0/2000 [00:00<?, ?it/s]


✓ Preprocessing completed!
  - Newly cached: 0 videos
  - Already cached: 2000 videos
  - Total time: 0.05 seconds (0.00 minutes)
  - Cache size: ~4000 MB

✓ Ready for training! The dataset will now load from cache.


# cache_manager.py

In [4]:
class CacheManager:
    """
    Dynamic Cache Manager with LRU (Least Recently Used) Eviction.

    Manages a bounded cache directory, automatically evicting oldest-accessed
    files when the cache exceeds the configured size limit. This enables
    training on datasets larger than available storage.

    Features:
    - Auto-detection: Automatically detects available storage and reserves 10GB
    - Warm caching: Pre-caches videos up to the limit before training starts
    - On-demand caching: Videos are cached only when first accessed
    - LRU eviction: Oldest-accessed files deleted first when limit exceeded
    - Statistics tracking: Hits, misses, evictions logged per epoch

    Performance Optimizations (v3):
    - Incremental size tracking: O(1) size checks instead of O(n) directory scans
    - OrderedDict LRU: O(1) access time updates and oldest file lookup
    - Batch evictions: Evict multiple files in one pass with 10% buffer
    - Lazy eviction: Only check size every N cache misses
    - Auto-detection: Calculates optimal cache size based on available storage
    """

    def __init__(self, cache_dir, max_size_gb=None, transforms=None, num_frames=30,
                 eviction_check_interval=10, eviction_buffer_percent=0.10,
                 auto_detect=True, reserve_gb=10.0):
        """
        Initialize the CacheManager.

        Args:
            cache_dir: Directory to store cached .npy files
            max_size_gb: Maximum cache size in gigabytes (None = auto-detect)
            transforms: Torchvision transforms to apply when caching
            num_frames: Number of frames per video (default: 30)
            eviction_check_interval: Check for eviction every N cache misses (default: 10)
            eviction_buffer_percent: Extra space to free during eviction (default: 0.10 = 10%)
            auto_detect: If True, auto-detect available storage (default: True)
            reserve_gb: GB to keep free when auto-detecting (default: 10.0)
        """
        self.cache_dir = cache_dir
        self.transforms = transforms
        self.num_frames = num_frames
        self.eviction_check_interval = eviction_check_interval
        self.eviction_buffer_percent = eviction_buffer_percent
        self.reserve_bytes = reserve_gb * (1024 ** 3)

        # Create cache directory if it doesn't exist
        os.makedirs(cache_dir, exist_ok=True)

        # Auto-detect or use provided max_size_gb
        if auto_detect and max_size_gb is None:
            self.max_size_bytes = self._calculate_max_cache_size()
            self.auto_detected = True
        else:
            self.max_size_bytes = (max_size_gb or 7.5) * (1024 ** 3)
            self.auto_detected = False

        # Incremental size tracking (Optimization 1)
        self.current_size_bytes = 0

        # LRU tracking with OrderedDict (Optimization 2)
        # Keys are filenames, values are file sizes for fast eviction
        # Order maintained: oldest at front, newest at back
        self.lru_cache = OrderedDict()

        # Statistics for current epoch
        self.hits = 0
        self.misses = 0
        self.evictions = 0
        self.warm_cached = 0  # Track videos cached during warm-up

        # Lazy eviction counter (Optimization 4)
        self.misses_since_eviction_check = 0

        # Initialize LRU cache and size from existing files
        self._initialize_from_disk()

        # Print initialization info
        detect_str = "(auto-detected)" if self.auto_detected else "(configured)"
        print(f"✓ CacheManager v3 initialized {detect_str}")
        print(f"  - Cache directory: {cache_dir}")
        print(f"  - Max cache size: {self.max_size_bytes / (1024**3):.1f} GB")
        print(f"  - Reserved free space: {reserve_gb:.1f} GB")
        print(f"  - Current cache size: {self.get_cache_size_gb():.2f} GB")
        print(f"  - Cached files: {len(self.lru_cache)}")
        print(f"  - Available to cache: {self._get_available_cache_space_gb():.2f} GB")

    def _calculate_max_cache_size(self):
        """
        Calculate maximum cache size based on available disk space.

        Returns:
            int: Maximum cache size in bytes (available - reserve)
        """
        try:
            disk_usage = shutil.disk_usage(self.cache_dir)
            available_bytes = disk_usage.free

            # Max cache = available space - reserve
            max_cache_bytes = available_bytes - self.reserve_bytes

            # Ensure we don't go negative
            max_cache_bytes = max(0, max_cache_bytes)

            print(f"✓ Storage auto-detection:")
            print(f"  - Total disk space: {disk_usage.total / (1024**3):.1f} GB")
            print(f"  - Currently free: {disk_usage.free / (1024**3):.1f} GB")
            print(f"  - Reserved: {self.reserve_bytes / (1024**3):.1f} GB")
            print(f"  - Available for cache: {max_cache_bytes / (1024**3):.1f} GB")

            return max_cache_bytes
        except Exception as e:
            print(f"⚠ Could not auto-detect storage: {e}")
            print(f"  Falling back to default 7.5 GB")
            return 7.5 * (1024 ** 3)

    def _get_available_cache_space_gb(self):
        """Get remaining space available for caching in GB."""
        return max(0, (self.max_size_bytes - self.current_size_bytes)) / (1024 ** 3)

    def _initialize_from_disk(self):
        """Load existing cache files and calculate initial size (runs once at startup)."""
        if os.path.exists(self.cache_dir):
            # Get all cached files with their modification times for initial LRU order
            cached_files = []
            for filename in os.listdir(self.cache_dir):
                if filename.endswith('.npy'):
                    filepath = os.path.join(self.cache_dir, filename)
                    file_size = os.path.getsize(filepath)
                    mtime = os.path.getmtime(filepath)
                    cached_files.append((filename, file_size, mtime))
                    self.current_size_bytes += file_size

            # Sort by modification time (oldest first) and populate OrderedDict
            cached_files.sort(key=lambda x: x[2])
            for filename, file_size, _ in cached_files:
                self.lru_cache[filename] = file_size

    def _format_bytes(self, size_bytes):
        """Format bytes to human-readable string."""
        for unit in ['B', 'KB', 'MB', 'GB']:
            if size_bytes < 1024:
                return f"{size_bytes:.2f} {unit}"
            size_bytes /= 1024
        return f"{size_bytes:.2f} TB"

    def update_max_size_from_disk(self):
        """
        Re-calculate max cache size based on current available disk space.
        Call this periodically to adapt to changing storage conditions.
        """
        old_max = self.max_size_bytes
        self.max_size_bytes = self._calculate_max_cache_size()

        if self.max_size_bytes != old_max:
            print(f"✓ Cache limit updated: {old_max/(1024**3):.1f} GB → {self.max_size_bytes/(1024**3):.1f} GB")

        return self.max_size_bytes

    def warm_cache(self, video_folder, video_list=None, max_videos=None):
        """
        Pre-cache videos up to the storage limit before training starts.

        This fills the cache with as many videos as possible, leaving the
        reserved free space (default 10GB) untouched. Videos are processed
        in order and caching stops when the limit is reached.

        Args:
            video_folder: Path to folder containing video files
            video_list: Optional list of video filenames to cache (in order)
                       If None, caches all .mp4 files in video_folder
            max_videos: Optional maximum number of videos to cache

        Returns:
            int: Number of videos cached during warm-up
        """
        print(f"\n{'='*50}")
        print(f"WARM CACHE: Pre-caching videos to fill available storage")
        print(f"{'='*50}")

        # Refresh max size based on current disk state
        if self.auto_detected:
            self.update_max_size_from_disk()

        # Get list of videos to potentially cache
        if video_list is None:
            video_list = [f for f in os.listdir(video_folder) if f.endswith('.mp4')]

        if max_videos is not None:
            video_list = video_list[:max_videos]

        # Filter out already cached videos
        uncached_videos = [v for v in video_list
                          if v.replace('.mp4', '.npy') not in self.lru_cache]

        print(f"  - Total videos available: {len(video_list)}")
        print(f"  - Already cached: {len(video_list) - len(uncached_videos)}")
        print(f"  - To be cached: {len(uncached_videos)}")
        print(f"  - Available space: {self._get_available_cache_space_gb():.2f} GB")
        print()

        cached_count = 0
        skipped_count = 0
        start_time = time.time()

        # Estimate average file size from existing cache or use default
        if len(self.lru_cache) > 0:
            avg_file_size = sum(self.lru_cache.values()) / len(self.lru_cache)
        else:
            # Estimate: 30 frames * 3 channels * 128 * 128 * 4 bytes = ~5.9 MB
            avg_file_size = 30 * 3 * 128 * 128 * 4

        for video_name in tqdm(uncached_videos, desc="Warm caching"):
            # Check if we have enough space for another video
            space_needed = avg_file_size * 1.1  # 10% buffer
            available_space = self.max_size_bytes - self.current_size_bytes

            if available_space < space_needed:
                skipped_count = len(uncached_videos) - cached_count
                print(f"\n⚠ Storage limit reached. Stopping warm cache.")
                print(f"  - Remaining space: {available_space / (1024**3):.2f} GB")
                print(f"  - Skipped: {skipped_count} videos")
                break

            # Cache this video
            video_path = os.path.join(video_folder, video_name)
            cache_filename = video_name.replace('.mp4', '.npy')
            cache_path = os.path.join(self.cache_dir, cache_filename)

            try:
                video_tensor = self._decode_video(video_path)
                np.save(cache_path, video_tensor.numpy())

                file_size = os.path.getsize(cache_path)
                self.lru_cache[cache_filename] = file_size
                self.current_size_bytes += file_size

                cached_count += 1
                self.warm_cached += 1

                # Update average file size estimate
                avg_file_size = (avg_file_size * 0.9) + (file_size * 0.1)

                del video_tensor

            except Exception as e:
                print(f"⚠ Failed to cache {video_name}: {e}")
                continue

        elapsed_time = time.time() - start_time

        print(f"\n{'='*50}")
        print(f"WARM CACHE COMPLETE")
        print(f"{'='*50}")
        print(f"  - Videos cached: {cached_count}")
        print(f"  - Time elapsed: {elapsed_time:.1f} seconds ({elapsed_time/60:.1f} minutes)")
        print(f"  - Cache size: {self.get_cache_size_gb():.2f} GB / {self.max_size_bytes/(1024**3):.1f} GB")
        print(f"  - Remaining free space: {self._get_available_cache_space_gb():.2f} GB")
        print(f"  - Disk free space: {shutil.disk_usage(self.cache_dir).free / (1024**3):.1f} GB")
        print()

        return cached_count

    def get_cache_size_bytes(self):
        """Get current cache size in bytes (O(1) - uses tracked value)."""
        return self.current_size_bytes

    def get_cache_size_gb(self):
        """Get current cache size in gigabytes."""
        return self.current_size_bytes / (1024 ** 3)

    def get_or_create(self, video_name, video_path):
        """
        Get cached tensor or create it from video.

        This is the main entry point. It:
        1. Checks if video is already cached (cache hit)
        2. If not, decodes video and caches it (cache miss)
        3. Periodically evicts old files if cache exceeds limit (lazy eviction)
        4. Updates LRU order using OrderedDict.move_to_end()

        Args:
            video_name: Name of the video file (e.g., 'video_001.mp4')
            video_path: Full path to the video file

        Returns:
            torch.Tensor: Video frames tensor [num_frames, 3, H, W]
        """
        cache_filename = video_name.replace('.mp4', '.npy')
        cache_path = os.path.join(self.cache_dir, cache_filename)

        # Check if already cached (CACHE HIT)
        if cache_filename in self.lru_cache:
            self.hits += 1
            # Move to end of OrderedDict (mark as recently used) - O(1)
            self.lru_cache.move_to_end(cache_filename)
            video_tensor = torch.from_numpy(np.load(cache_path))
            return video_tensor

        # Not cached - decode from video (CACHE MISS)
        self.misses += 1
        self.misses_since_eviction_check += 1
        video_tensor = self._decode_video(video_path)

        # Save to cache
        np.save(cache_path, video_tensor.numpy())

        # Get file size and update tracking
        file_size = os.path.getsize(cache_path)
        self.lru_cache[cache_filename] = file_size  # Add to end (most recent)
        self.current_size_bytes += file_size

        # Lazy eviction: Only check every N misses (Optimization 4)
        if self.misses_since_eviction_check >= self.eviction_check_interval:
            self._batch_evict_if_needed()
            self.misses_since_eviction_check = 0

        return video_tensor

    def _decode_video(self, video_path):
        """Decode video file to tensor (called on cache miss)."""
        cap = cv2.VideoCapture(video_path)
        frames = []

        for _ in range(self.num_frames):
            ret, frame = cap.read()
            if not ret:
                # Pad with black frame if video is shorter
                frame_tensor = torch.zeros((3, HEIGHT, WIDTH))
            else:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                if self.transforms:
                    frame = Image.fromarray(frame)
                    frame_tensor = self.transforms(frame)
                else:
                    frame_tensor = torch.from_numpy(frame).permute(2, 0, 1).float() / 255.0

            frames.append(frame_tensor)

        cap.release()

        # Stack frames: [num_frames, 3, H, W]
        video_tensor = torch.stack(frames, dim=0)
        del frames

        return video_tensor

    def _batch_evict_if_needed(self):
        """
        Batch eviction with buffer (Optimizations 3 & 4).

        Instead of evicting one file at a time with size checks between each,
        this calculates how much space to free (including a buffer) and
        evicts multiple files in one pass.
        """
        if self.current_size_bytes <= self.max_size_bytes:
            return  # No eviction needed

        # Calculate how much to evict (current excess + buffer)
        excess_bytes = self.current_size_bytes - self.max_size_bytes
        buffer_bytes = self.max_size_bytes * self.eviction_buffer_percent
        bytes_to_evict = excess_bytes + buffer_bytes

        # Collect files to evict (oldest first from OrderedDict)
        bytes_evicted = 0
        files_to_remove = []

        for filename, file_size in self.lru_cache.items():
            if bytes_evicted >= bytes_to_evict:
                break
            files_to_remove.append((filename, file_size))
            bytes_evicted += file_size

        # Batch remove files
        for filename, file_size in files_to_remove:
            filepath = os.path.join(self.cache_dir, filename)
            if os.path.exists(filepath):
                os.remove(filepath)
            del self.lru_cache[filename]
            self.current_size_bytes -= file_size
            self.evictions += 1

    def get_stats(self):
        """
        Get cache statistics for current epoch.

        Returns:
            dict: Statistics including hits, misses, hit_rate, evictions, cache_size
        """
        total_accesses = self.hits + self.misses
        hit_rate = (self.hits / total_accesses * 100) if total_accesses > 0 else 0

        return {
            'hits': self.hits,
            'misses': self.misses,
            'hit_rate': hit_rate,
            'evictions': self.evictions,
            'cache_size_gb': self.get_cache_size_gb(),
            'num_cached_files': len(self.lru_cache),
            'warm_cached': self.warm_cached,
            'available_gb': self._get_available_cache_space_gb()
        }

    def reset_epoch_stats(self):
        """Reset hit/miss/eviction counters for new epoch."""
        self.hits = 0
        self.misses = 0
        self.evictions = 0
        self.misses_since_eviction_check = 0

    def print_stats(self, epoch=None):
        """Print formatted cache statistics."""
        stats = self.get_stats()
        epoch_str = f"Epoch {epoch} " if epoch is not None else ""
        print(f"{epoch_str}Cache Stats: "
              f"Hits: {stats['hits']} | "
              f"Misses: {stats['misses']} | "
              f"Hit Rate: {stats['hit_rate']:.1f}% | "
              f"Evictions: {stats['evictions']} | "
              f"Size: {stats['cache_size_gb']:.2f}/{self.max_size_bytes/(1024**3):.1f} GB")

# Initialize CacheManager with preprocessing transforms
cache_transforms = transforms.Compose([
    transforms.Resize((HEIGHT, WIDTH)),
    transforms.ToTensor()
])

# Configuration - Auto-detect available storage with 10GB reserve
RESERVE_GB = 10.0  # Keep this much free space on disk

cache_manager = CacheManager(
    cache_dir=CACHE_DIR,
    max_size_gb=None,                # None = auto-detect from available storage
    transforms=cache_transforms,
    num_frames=SEQ_LEN,
    eviction_check_interval=10,      # Check every 10 cache misses
    eviction_buffer_percent=0.10,    # Free 10% extra space when evicting
    auto_detect=True,                # Enable storage auto-detection
    reserve_gb=RESERVE_GB            # Keep 10GB free space
)

# OPTIONAL: Pre-cache videos to fill available storage
# cache_manager.warm_cache(VIDEO_DIR)

✓ Storage auto-detection:
  - Total disk space: 112.6 GB
  - Currently free: 59.9 GB
  - Reserved: 10.0 GB
  - Available for cache: 49.9 GB
✓ CacheManager v3 initialized (auto-detected)
  - Cache directory: /content/cache/
  - Max cache size: 49.9 GB
  - Reserved free space: 10.0 GB
  - Current cache size: 10.99 GB
  - Cached files: 2000
  - Available to cache: 38.96 GB


# conv_lstm_classifier.py

In [5]:
class ConvLSTMCell(nn.Module):
    """
    The Single Memory Unit of the video.
    """

    def __init__(self, input_dim, hidden_dim, kernel_size, bias):
        super(ConvLSTMCell, self).__init__()

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim

        self.kernel_size = kernel_size
        self.padding = kernel_size[0] // 2, kernel_size[1] // 2
        self.bias = bias

        self.conv = nn.Conv2d(in_channels=self.input_dim + self.hidden_dim,
                              out_channels=4 * self.hidden_dim,
                              kernel_size=self.kernel_size,
                              padding=self.padding,
                              bias=self.bias)

    def forward(self, input_tensor, cur_state):
        h_cur, c_cur = cur_state

        combined = torch.cat([input_tensor, h_cur], dim=1)
        combined_conv = self.conv(combined)

        cc_i, cc_f, cc_o, cc_g = torch.split(combined_conv, self.hidden_dim, dim=1)
        i = torch.sigmoid(cc_i)
        f = torch.sigmoid(cc_f)
        o = torch.sigmoid(cc_o)
        g = torch.tanh(cc_g)

        c_next = f * c_cur + i * g
        h_next = o * torch.tanh(c_next)

        return h_next, c_next

    def init_hidden(self, batch_size, image_size):
        height, width = image_size
        return (torch.zeros(batch_size, self.hidden_dim, height, width, device=self.conv.weight.device),
                torch.zeros(batch_size, self.hidden_dim, height, width, device=self.conv.weight.device))

class ConvLSTM(nn.Module):
    """
    The Observer of the video
    """

    def __init__(self, input_dim, hidden_dim, kernel_size, num_layers,
                 batch_first=True, bias=True, return_all_layers=False): # Changed default to batch_first=True
        super(ConvLSTM, self).__init__()

        self._check_kernel_size_consistency(kernel_size)
        kernel_size = self._extend_for_multilayer(kernel_size, num_layers)
        hidden_dim = self._extend_for_multilayer(hidden_dim, num_layers)

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.kernel_size = kernel_size
        self.num_layers = num_layers
        self.batch_first = batch_first
        self.bias = bias
        self.return_all_layers = return_all_layers

        cell_list = []
        for i in range(0, self.num_layers):
            cur_input_dim = self.input_dim if i == 0 else self.hidden_dim[i - 1]
            cell_list.append(ConvLSTMCell(input_dim=cur_input_dim,
                                          hidden_dim=self.hidden_dim[i],
                                          kernel_size=self.kernel_size[i],
                                          bias=self.bias))

        self.cell_list = nn.ModuleList(cell_list)

    def forward(self, input_tensor, hidden_state=None):
        # input_tensor format: [Batch, Time, Channel, Height, Width]

        if not self.batch_first:
            # (Time, Batch, Channel, Height, Width) -> (Batch, Time, Channel, Height, Width)
            input_tensor = input_tensor.permute(1, 0, 2, 3, 4)

        # Get Dimensions using the Correct Indices
        b, seq_len, _, h, w = input_tensor.size()

        if hidden_state is None:
            hidden_state = self._init_hidden(batch_size=b, image_size=(h, w))

        layer_output_list = []
        last_state_list = []

        cur_layer_input = input_tensor

        for layer_idx in range(self.num_layers):
            h, c = hidden_state[layer_idx]
            output_inner = []

            # Loop over TIME (seq_len), not Batch
            for t in range(seq_len):
                # Slice the time dimension: [Batch, Channel, H, W]
                # If layer_idx == 0, cur_layer_input is [B, T, C, H, W]
                # If layer_idx > 0, cur_layer_input is [B, T, Hidden, H, W] (from previous layer stack)

                input_t = cur_layer_input[:, t, :, :, :]

                h, c = self.cell_list[layer_idx](input_tensor=input_t, cur_state=[h, c])
                output_inner.append(h)

            # Stack along Time dimension (dim=1 because we enforce batch_first internally now)
            layer_output = torch.stack(output_inner, dim=1)
            cur_layer_input = layer_output

            layer_output_list.append(layer_output)
            last_state_list.append([h, c])

        if not self.return_all_layers:
            layer_output_list = layer_output_list[-1:]
            last_state_list = last_state_list[-1:]

        return layer_output_list, last_state_list

    def _init_hidden(self, batch_size, image_size):
        init_states = []
        for i in range(self.num_layers):
            init_states.append(self.cell_list[i].init_hidden(batch_size, image_size))
        return init_states

    @staticmethod
    def _check_kernel_size_consistency(kernel_size):
        if not (isinstance(kernel_size, tuple) or
                (isinstance(kernel_size, list) and all([isinstance(elem, tuple) for elem in kernel_size]))):
            raise ValueError('`kernel_size` must be tuple or list of tuples')

    @staticmethod
    def _extend_for_multilayer(param, num_layers):
        if not isinstance(param, list):
            param = [param] * num_layers
        return param


class ConvLSTMModel(nn.Module):
    """
    The Judge of the video.
    """
    def __init__(self, input_dim, hidden_dim, kernel_size, num_layers, height, width,
                 batch_first=True, bias=True, return_all_layers=False, num_classes=3, dropout_rate=0.5):
        super(ConvLSTMModel, self).__init__()

        # Ensure batch_first is passed correctly
        self.convlstm = ConvLSTM(input_dim, hidden_dim, kernel_size, num_layers,
                                 batch_first=batch_first, bias=bias,
                                 return_all_layers=return_all_layers)

        # Dropout for regularization (prevents overfitting)
        self.dropout = nn.Dropout(p=dropout_rate)

        # Input to linear is: Hidden_Dim * H * W
        self.linear = nn.Linear(hidden_dim[-1] * height * width, num_classes)

    def forward(self, input_tensor, hidden_state=None):
        x, _ = self.convlstm(input_tensor)

        # x[0] shape is now guaranteed to be [Batch, Time, Hidden, H, W]
        # We take the last time step: x[0][:, -1, :, :, :]

        last_time_step = x[0][:, -1, :, :, :]

        # Flatten starting from dimension 1 (Channels/Hidden)
        flattened = torch.flatten(last_time_step, start_dim=1)

        # Apply dropout for regularization (only active during training)
        flattened = self.dropout(flattened)

        output = self.linear(flattened)
        return output

# dataset.py

In [6]:
class MVOVideoDataset(Dataset):
    """
    This takes a 3-second video and turns it into
    a 'data packet' for the AI to study.

    Now with dynamic caching support via CacheManager:
    - On-demand caching: Videos cached only when first accessed
    - LRU eviction: Old cache files automatically deleted when limit exceeded
    - Bounded storage: Works with datasets larger than available storage
    """
    def __init__(self, video_folder, label_folder, transforms=None, cache_manager=None, cache_folder=None):
        """
        Initialize the dataset.

        Args:
            video_folder: Path to video files
            label_folder: Path to label CSV files
            transforms: Torchvision transforms (used if no cache_manager)
            cache_manager: CacheManager instance for dynamic caching (preferred)
            cache_folder: Legacy cache folder path (for backward compatibility)
        """
        self.video_folder = video_folder
        self.label_folder = label_folder
        self.cache_folder = cache_folder
        self.transforms = transforms
        self.cache_manager = cache_manager
        self.use_cache = False

        valid_video_files = []
        try:
            all_video_files = [f for f in os.listdir(video_folder) if f.endswith('.mp4')]
            for video_name in all_video_files:
                video_path = os.path.join(video_folder, video_name)

                # Check for corresponding CSV file (same name, just .csv instead of .mp4)
                csv_name = video_name.replace('.mp4', '.csv')
                csv_path = os.path.join(label_folder, csv_name)

                if os.path.exists(video_path) and os.path.exists(csv_path):
                    valid_video_files.append(video_name)
                else:
                    if not os.path.exists(video_path):
                        print(f"WARNING: Video file not found: {video_path}. Skipping.")
                    if not os.path.exists(csv_path):
                        print(f"WARNING: Label file not found for video {video_name}. Skipping.")

            self.video_files = valid_video_files

            # Determine caching mode
            if cache_manager is not None:
                print(f"✓ Using CacheManager for dynamic on-demand caching")
                self.use_cache = True
            elif cache_folder and os.path.exists(cache_folder):
                # Legacy: Check for pre-cached files
                cached_count = sum(1 for f in self.video_files
                                  if os.path.exists(os.path.join(cache_folder, f.replace('.mp4', '.npy'))))
                if cached_count > 0:
                    self.use_cache = True
                    print(f"✓ Using static cached frames from: {cache_folder}")
                    print(f"  ({cached_count}/{len(self.video_files)} videos cached)")

            if not self.use_cache and cache_manager is None:
                print(f"⚠ Loading frames from videos directly (no cache - slower)")

            print(f"Initialized dataset with {len(self.video_files)} valid video-label pairs.")

        except FileNotFoundError:
            print(f"Error: Could not find folder {video_folder} or {label_folder}.")
            self.video_files = []

    def __len__(self):
        return len(self.video_files)

    def __getitem__(self, idx):
        video_name = self.video_files[idx]
        video_path = os.path.join(self.video_folder, video_name)

        # Use CacheManager for dynamic caching
        if self.cache_manager is not None:
            video_tensor = self.cache_manager.get_or_create(video_name, video_path)

        # Load from static cache if available
        elif self.use_cache and self.cache_folder:
            cache_path = os.path.join(self.cache_folder, video_name.replace('.mp4', '.npy'))
            if os.path.exists(cache_path):
                video_tensor = torch.from_numpy(np.load(cache_path))
            else:
                video_tensor = self._load_from_video(video_name)

        # Decode from video directly
        else:
            video_tensor = self._load_from_video(video_name)

        # Get the Label from the matching CSV file
        csv_name = video_name.replace('.mp4', '.csv')
        csv_path = os.path.join(self.label_folder, csv_name)

        df = pd.read_csv(csv_path)
        label = self.labeler(df)

        return video_tensor, torch.tensor(label).long()

    def _load_from_video(self, video_name):
        """Load and process frames from video file (used when cache is not available)"""
        video_path = os.path.join(self.video_folder, video_name)

        cap = cv2.VideoCapture(video_path)
        frames = []
        for _ in range(30):
            ret, frame = cap.read()
            if not ret:
                # If a video is shorter than 3s, pad with a black frame
                frame_tensor = torch.zeros((3, HEIGHT, WIDTH))
            else:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                if self.transforms:
                    frame = Image.fromarray(frame)
                    frame_tensor = self.transforms(frame)
                else:
                    # Fallback if no transforms are provided
                    frame_tensor = torch.from_numpy(frame).permute(2, 0, 1).float() / 255.0

            frames.append(frame_tensor)

        cap.release()

        # Convert list to tensor [30, 3, 128, 128]
        video_tensor = torch.stack(frames, dim=0)

        # Memory cleanup
        del frames

        return video_tensor

    def labeler(self, df):
        df_lbl_count = []
        # Logic to handle counts for classes 0, 1, and 2
        # Added a check for the column name to prevent KeyErrors
        col = 'label_id_corrected' if 'label_id_corrected' in df.columns else df.columns[-1]
        counts = df[col].tail(24).value_counts()

        for i in range(0, 3):
            df_lbl_count.append(counts.get(i, 0))

        if df_lbl_count[0] == 24:
            label = 0 # Front
        elif df_lbl_count[1] > df_lbl_count[2]:
            label = 1 # Left
        elif df_lbl_count[1] < df_lbl_count[2]:
            label = 2 # Right
        else:
            label = df[col].tail(12).mode()[0]

        return label

    def class_counter(self):
        label_counts = {0: 0, 1: 0, 2: 0}
        for video_name in self.video_files:
            csv_name = video_name.replace('.mp4', '.csv')
            csv_path = os.path.join(self.label_folder, csv_name)

            df = pd.read_csv(csv_path)

            label = self.labeler(df)
            label_counts[label] += 1
        return label_counts, sum(label_counts.values())

# train.py

In [7]:
# Configurations
BATCH = 7
NUM_EPOCHS = 1
LEARNING_RATE = 1e-4
SAVED_MODEL_PATH = "best_convlstm.pth"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Gradient Accumulation Configuration
ACCUMULATION_STEPS = 4  # Effective batch size = BATCH * ACCUMULATION_STEPS

# Early Stopping Configuration
EARLY_STOP_PATIENCE = 5  # Stop if no improvement for 5 epochs
MIN_DELTA = 0.01  # Minimum change to qualify as improvement (0.01%)

# Setting a fixed random seed to ensure that
# we get the exact same data split every time we run the script
SEED = 8
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# Model Parameters
PARAMS = {
    'input_dim': 3,
    'hidden_dim': [64, 32],
    'kernel_size': (3, 3),
    'num_layers': 2,
    'height': HEIGHT,
    'width': WIDTH,
    'num_classes': 3
}

def train_one_epoch(model, loader, criterion, optimizer, accumulation_steps=1, max_grad_norm=1.0):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    total_grad_norm = 0.0
    optimizer.zero_grad()  # Zero gradients at the start

    loop = tqdm(loader, leave=True)
    for batch_idx, (data, targets) in enumerate(loop):
        data = data.float().to(DEVICE)
        targets = targets.to(DEVICE)

        # Forward
        scores = model(data)
        loss = criterion(scores, targets)

        # Scale loss by accumulation steps (important for correct gradient magnitude)
        loss = loss / accumulation_steps

        # Backward (accumulate gradients)
        loss.backward()

        # Only update weights every accumulation_steps batches
        if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(loader):
            # Gradient Clipping: Prevent exploding gradients in ConvLSTM
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            total_grad_norm += grad_norm.item()

            optimizer.step()
            optimizer.zero_grad()

            # Update progress bar on weight updates
            loop.set_description(f"Loss: {loss.item()*accumulation_steps:.4f} | GradNorm: {grad_norm.item():.3f} | Step: {(batch_idx+1)//accumulation_steps}")

        # Stats (track unscaled loss for reporting)
        running_loss += loss.item() * accumulation_steps
        _, predictions = scores.max(1)
        correct += (predictions == targets).sum().item()
        total += targets.size(0)

        # Memory cleanup: Delete intermediate tensors
        del data, targets, scores, loss, predictions

    # Clear GPU cache after epoch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    # Average gradient norm per weight update (not per batch)
    num_updates = (len(loader) + accumulation_steps - 1) // accumulation_steps
    avg_grad_norm = total_grad_norm / num_updates
    return running_loss / len(loader), 100 * correct / total, avg_grad_norm

def main():
    # Data Setup - Using CacheManager for dynamic on-demand caching
    # Note: transforms are handled by cache_manager, not the dataset
    transforms_train = transforms.Compose([
        transforms.Resize((HEIGHT, WIDTH)),
        transforms.ToTensor()
    ])

    # Use the global cache_manager initialized in cache_manager.py cell
    full_dataset = MVOVideoDataset(
        VIDEO_DIR,
        LABEL_DIR,
        transforms=transforms_train,
        cache_manager=cache_manager
    )

    # Train/Val/Test Split
    total_size = len(full_dataset)
    train_size = int(0.6 * total_size)                  # 60% for Training
    val_size = int(0.2 * total_size)                    # 20% for Validation
    test_size = total_size - train_size - val_size      # Remaining 20% for Testing

    # The generator with our fixed seed so the split is always the same
    generator = torch.Generator().manual_seed(SEED)

    # Split the dataset into three parts
    train_dataset, val_dataset, _ = random_split(
        full_dataset,
        [train_size, val_size, test_size],
        generator=generator
    )
    # Note: The last part is '_' because we don't touch the test set in train.py

    print(f"Data Split -> Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test (Unused): {test_size}")

    train_loader = DataLoader(train_dataset, batch_size=BATCH, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH, shuffle=False, num_workers=0)

    # Model Setup
    model = ConvLSTMModel(
        input_dim=PARAMS['input_dim'],
        hidden_dim=PARAMS['hidden_dim'],
        kernel_size=PARAMS['kernel_size'],
        num_layers=PARAMS['num_layers'],
        height=PARAMS['height'],
        width=PARAMS['width'],
        num_classes=PARAMS['num_classes']
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # Learning Rate Scheduler: Reduces LR when validation accuracy plateaus
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',           # Maximize validation accuracy
        factor=0.5,           # Reduce LR by half
        patience=3,           # Wait 3 epochs before reducing
        min_lr=1e-7           # Don't go below this LR
    )

    # Training Loop with Early Stopping
    best_acc = 0
    epochs_no_improve = 0
    early_stop = False
    print(f"Training on {DEVICE} with {len(train_dataset)} videos.")
    print(f"Gradient Accumulation: {ACCUMULATION_STEPS} steps (Effective batch size: {BATCH * ACCUMULATION_STEPS})")
    print(f"Early stopping enabled: patience={EARLY_STOP_PATIENCE}, min_delta={MIN_DELTA}%")
    print(f"Dynamic caching enabled with LRU eviction")

    for epoch in range(NUM_EPOCHS):
        # Reset cache statistics for this epoch
        cache_manager.reset_epoch_stats()

        print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
        train_loss, train_acc, avg_grad_norm = train_one_epoch(
            model, train_loader, criterion, optimizer,
            accumulation_steps=ACCUMULATION_STEPS, max_grad_norm=1.0
        )

        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        val_loss = 0.0
        val_latencies = []

        with torch.no_grad():
            for batch_idx, (x, y) in enumerate(val_loader):
                x = x.float().to(DEVICE)
                y = y.to(DEVICE)

                # Measure inference time
                if DEVICE.type == 'cuda':
                    torch.cuda.synchronize()

                start_time = time.perf_counter()
                scores = model(x)

                if DEVICE.type == 'cuda':
                    torch.cuda.synchronize()

                end_time = time.perf_counter()

                # Calculate validation loss
                batch_loss = criterion(scores, y)
                val_loss += batch_loss.item()

                # Skip first batch for warm-up
                if batch_idx >= 1:
                    val_latencies.append(end_time - start_time)

                _, preds = scores.max(1)
                val_correct += (preds == y).sum().item()
                val_total += y.size(0)

                # Memory cleanup: Delete intermediate tensors
                del x, y, scores, preds, batch_loss

        # Clear GPU cache after validation
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        val_acc = 100 * val_correct / val_total
        val_loss_avg = val_loss / len(val_loader)
        avg_val_latency_ms = (np.mean(val_latencies) * 1000) if len(val_latencies) > 0 else 0
        val_throughput = (1 / np.mean(val_latencies)) if len(val_latencies) > 0 else 0
        current_lr = optimizer.param_groups[0]['lr']

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")
        print(f"Avg Gradient Norm: {avg_grad_norm:.4f} (clipped at 1.0)")
        print(f"Val Inference: {avg_val_latency_ms:.2f} ms/batch | {val_throughput:.2f} batches/sec")
        print(f"Current LR: {current_lr:.2e}")

        # Print cache statistics for this epoch
        cache_manager.print_stats(epoch=epoch+1)

        # Update learning rate based on validation accuracy
        scheduler.step(val_acc)

        # Save Best Model & Early Stopping Check
        if val_acc > best_acc + MIN_DELTA:
            best_acc = val_acc
            epochs_no_improve = 0
            torch.save(model.state_dict(), SAVED_MODEL_PATH)
            print(f"✓ New best model saved! ({val_acc:.2f}%)")
        else:
            epochs_no_improve += 1
            print(f"No improvement for {epochs_no_improve} epoch(s). Best: {best_acc:.2f}%")

            if epochs_no_improve >= EARLY_STOP_PATIENCE:
                print(f"\n⚠ Early stopping triggered! No improvement for {EARLY_STOP_PATIENCE} epochs.")
                print(f"Best validation accuracy: {best_acc:.2f}%")
                early_stop = True
                break

    if not early_stop:
        print(f"\n✓ Training completed all {NUM_EPOCHS} epochs.")
    print(f"Final best validation accuracy: {best_acc:.2f}%")

if __name__ == "__main__":
    main()


✓ Using CacheManager for dynamic on-demand caching
Initialized dataset with 2000 valid video-label pairs.
Data Split -> Train: 1200 | Val: 400 | Test (Unused): 400
Training on cuda with 1200 videos.
Gradient Accumulation: 4 steps (Effective batch size: 28)
Early stopping enabled: patience=5, min_delta=0.01%
Dynamic caching enabled with LRU eviction

Epoch 1/1


  0%|          | 0/172 [00:00<?, ?it/s]

Train Loss: 1.0940 | Train Acc: 38.50% | Val Acc: 42.25%
Avg Gradient Norm: 2.2527 (clipped at 1.0)
Val Inference: 683.19 ms/batch | 1.46 batches/sec
Current LR: 1.00e-04
Epoch 1 Cache Stats: Hits: 1600 | Misses: 0 | Hit Rate: 100.0% | Evictions: 0 | Size: 10.99/49.9 GB
✓ New best model saved! (42.25%)

✓ Training completed all 1 epochs.
Final best validation accuracy: 42.25%


# tester.py

In [9]:
class Tester:
    """
    It loads a trained model, feeds it unseen data,
    and records how accurately and how fast the model makes decisions.
    """
    def __init__(self, model_path, device):
        self.model_path = model_path
        self.device = device
        self.transforms = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((HEIGHT, WIDTH))
        ])

        # Load the Architecture
        self.model = ConvLSTMModel(
            input_dim=3,
            hidden_dim=[64, 32],
            kernel_size=(3, 3),
            num_layers=2,
            height=HEIGHT,
            width=WIDTH,
            num_classes=3,
            dropout_rate=0.5  # Dropout not applied during eval mode
        ).to(self.device)

        # Load the Weights
        self.load_weights()

    def load_weights(self):
        # Attempts to load the best model file and sets it to evaluation mode
        if os.path.exists(self.model_path):
            print(f"Loading model from {self.model_path}...")
            self.model.load_state_dict(torch.load(self.model_path, map_location=self.device))
            self.model.eval()
        else:
            raise FileNotFoundError(f"Model file not found at {self.model_path}. Did you run train.py?")

    def test(self):
        # The main evaluation loop.
        print("Preparing Test Data...")
        # Use cache_manager for dynamic caching during testing
        full_dataset = MVOVideoDataset(
            VIDEO_DIR,
            LABEL_DIR,
            transforms=self.transforms,
            cache_manager=cache_manager
        )

        # Re-creating the 60/20/20 split
        SEED = 8
        total_size = len(full_dataset)
        train_size = int(0.6 * total_size)
        val_size = int(0.2 * total_size)
        test_size = total_size - train_size - val_size

        generator = torch.Generator().manual_seed(SEED)

        # Split, but this time we only care about the LAST chunk (test_dataset)
        _, _, test_dataset = random_split(
            full_dataset,
            [train_size, val_size, test_size],
            generator=generator
        )
        # Note that the first two are "_" because we do not need them

        test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)

        print(f"Evaluating {len(test_dataset)} videos and measuring latency...")

        all_preds = []
        all_labels = []
        latencies = []

        # Evaluation Loop
        with torch.no_grad():
            for i, (video_tensor, labels) in enumerate(tqdm(test_loader, leave=True)):
                video_tensor = video_tensor.float().to(self.device)
                labels = labels.to(self.device)

                # Latency Measurement Start
                if self.device.type == 'cuda':
                    torch.cuda.synchronize()

                start_time = time.perf_counter() # Timer

                outputs = self.model(video_tensor) # Forward pass (The Inference)

                if self.device.type == 'cuda':
                    torch.cuda.synchronize() # Wait for the GPU to finish the math

                end_time = time.perf_counter()
                # Latency Measurement End

                # We skip the first 5 frames ('warm-up')
                if i >= 5:
                    latencies.append(end_time - start_time)

                # Convert raw scores to the predicted class index (0, 1, or 2)
                _, predicted = torch.max(outputs.data, 1)

                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

                # Memory cleanup: Delete intermediate tensors
                del video_tensor, labels, outputs, predicted

                # Periodic cache clearing every 50 videos
                if i % 50 == 0:
                    if self.device.type == 'cuda':
                        torch.cuda.empty_cache()
                    gc.collect()

        # Calculate Latency Stats
        avg_latency_ms = np.mean(latencies) * 1000 if len(latencies) > 0 else 0
        inf_fps = 1 / np.mean(latencies) if len(latencies) > 0 else 0

        # Calculate and Print all results
        self.calculate_metrics(all_labels, all_preds, avg_latency_ms, inf_fps)

        # Save detailed logs to a CSV
        self.save_results(all_labels, all_preds)

    def calculate_metrics(self, y_true, y_pred, avg_latency_ms, inf_fps):
        # Computes statistical performance and prints the Final Report
        print(f"Avg Latency:        {avg_latency_ms:.2f} ms per video clip")
        print(f"Inference Speed:    {inf_fps:.2f} clips per second")
        # Computes statistical performance and prints the Final Report
        print("\n" + "-"*40)
        print("       FINAL PERFORMANCE REPORT       ")
        print("-"*40)

        # Accuracy
        acc = accuracy_score(y_true, y_pred)
        print(f"Overall Accuracy:   {acc*100:.2f}%")

        # Precision and Recall
        precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
        recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)

        print(f"Precision:          {precision:.4f}")
        print(f"Recall:             {recall:.4f}")
        print("-" * 40)

        print(f"Avg Latency:        {avg_latency_ms:.2f} ms per video clip")
        print(f"Inference Speed:    {inf_fps:.2f} clips per second")

        print("-" * 40)
        print("Detailed Class Report:")
        # Generates a table for Front(0), Left(1), Right(2)
        print(classification_report(y_true, y_pred, target_names=['Front', 'Left', 'Right'], zero_division=0))

    def save_results(self, y_true, y_pred):
        # Creates a CSV to see exactly which videos failed
        df = pd.DataFrame({
            'Actual_Label': y_true,
            'Predicted_Label': y_pred
        })

        # Map numbers back to words for readability
        label_map = {0: 'Front', 1: 'Left', 2: 'Right'}
        df['Actual_Text'] = df['Actual_Label'].map(label_map)
        df['Predicted_Text'] = df['Predicted_Label'].map(label_map)

        # Check if correct
        df['Correct'] = df['Actual_Label'] == df['Predicted_Label']

        save_path = "test_results.csv"
        df.to_csv(save_path, index=False)
        print(f"\nDetailed predictions saved to '{save_path}'")

if __name__ == "__main__":
    # Configuration
    MODEL_PATH = "best_convlstm.pth"
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Run the Tester
    tester = Tester(MODEL_PATH, DEVICE)
    tester.test()


Loading model from best_convlstm.pth...
Preparing Test Data...
✓ Using CacheManager for dynamic on-demand caching
Initialized dataset with 2000 valid video-label pairs.
Evaluating 400 videos and measuring latency...


  0%|          | 0/400 [00:00<?, ?it/s]

Avg Latency:        85.84 ms per video clip
Inference Speed:    11.65 clips per second

----------------------------------------
       FINAL PERFORMANCE REPORT       
----------------------------------------
Overall Accuracy:   41.25%
Precision:          0.6065
Recall:             0.4125
----------------------------------------
Avg Latency:        85.84 ms per video clip
Inference Speed:    11.65 clips per second
----------------------------------------
Detailed Class Report:
              precision    recall  f1-score   support

       Front       0.47      0.49      0.48       134
        Left       0.38      0.71      0.50       139
       Right       1.00      0.01      0.02       127

    accuracy                           0.41       400
   macro avg       0.62      0.40      0.33       400
weighted avg       0.61      0.41      0.34       400


Detailed predictions saved to 'test_results.csv'
Loading model from best_convlstm.pth...
Preparing Test Data...
✓ Using CacheManager for

  0%|          | 0/400 [00:00<?, ?it/s]

Avg Latency:        75.66 ms per video clip
Inference Speed:    13.22 clips per second

----------------------------------------
       FINAL PERFORMANCE REPORT       
----------------------------------------
Overall Accuracy:   41.25%
Precision:          0.6065
Recall:             0.4125
----------------------------------------
Avg Latency:        75.66 ms per video clip
Inference Speed:    13.22 clips per second
----------------------------------------
Detailed Class Report:
              precision    recall  f1-score   support

       Front       0.47      0.49      0.48       134
        Left       0.38      0.71      0.50       139
       Right       1.00      0.01      0.02       127

    accuracy                           0.41       400
   macro avg       0.62      0.40      0.33       400
weighted avg       0.61      0.41      0.34       400


Detailed predictions saved to 'test_results.csv'
